#### 1. Build a list of dataframe from loose csv files in folder and combine

In [1]:
import glob
import pandas as pandas

csv_folder = "../data/loose_csv/*.csv"
file_paths = glob.glob(csv_folder)

print(f"Found {len(file_paths)} files")

dataframe_list = [pandas.read_csv(file) for file in file_paths]
dataframe_combined = pandas.concat(dataframe_list, ignore_index=True)
dataframe_combined = dataframe_combined.reset_index(drop=True)
dataframe_combined.dropna()

Found 15 files


,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Protocol,Timestamp,Flow Duration,Total Fwd Packet,Total Bwd packets,...,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Attack Name,Label
0,34.173.20.6-192.168.137.234-443-58644-6,34.173.20.6,443,192.168.137.234,58644,6,14/09/2023 09:24:26 AM,14746364,6,0,...,0.0,0.0,0.0,0.0,14681116.0,0.000000,14681116.0,14681116.0,Benign Traffic,0
1,34.173.20.6-192.168.137.234-443-58646-6,34.173.20.6,443,192.168.137.234,58646,6,14/09/2023 09:24:41 AM,60102110,18,0,...,219123.0,0.0,219123.0,219123.0,14957706.0,190543.248665,15061923.0,14672051.0,Benign Traffic,0
2,52.40.210.103-192.168.137.40-18665-60378-6,52.40.210.103,18665,192.168.137.40,60378,6,14/09/2023 09:24:14 AM,90106769,9,0,...,582529.0,0.0,582529.0,582529.0,29841412.0,319090.538141,30049066.0,29473997.0,Benign Traffic,0
3,10.0.0.3-10.0.0.254-44505-1883-6,10.0.0.3,44505,10.0.0.254,1883,6,14/09/2023 09:24:15 AM,119998980,123,123,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,Benign Traffic,0
4,10.0.0.7-10.0.0.254-45727-1883-6,10.0.0.7,45727,10.0.0.254,1883,6,14/09/2023 09:24:15 AM,119998539,123,123,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,Benign Traffic,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3385308,192.168.137.250-192.168.137.156-1883-58194-6,192.168.137.250,1883,192.168.137.156,58194,6,29/08/2023 05:05:46 PM,0,2,0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,MQTT DDoS Publish Flood,1
3385309,192.168.137.250-192.168.137.156-1883-58232-6,192.168.137.250,1883,192.168.137.156,58232,6,29/08/2023 05:05:46 PM,1552,2,0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,MQTT DDoS Publish Flood,1
3385310,192.168.137.250-192.168.137.156-1883-58202-6,192.168.137.250,1883,192.168.137.156,58202,6,29/08/2023 05:05:46 PM,22717,2,0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,MQTT DDoS Publish Flood,1
3385311,192.168.137.250-192.168.137.156-1883-58206-6,192.168.137.250,1883,192.168.137.156,58206,6,29/08/2023 05:05:46 PM,0,2,0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,MQTT DDoS Publish Flood,1


#### 1a. Filter or combine problem attack targets

In [2]:
dataframe_filtered_rps = dataframe_combined[dataframe_combined['Attack Name'] != 'Recon Ping Sweep']
dataframe_filtered = dataframe_filtered_rps[dataframe_filtered_rps['Attack Name'] != 'MITM ARP Spoofing'].copy()

attack_mapping = {
    'DDoS UDP Flood': 'UDP Flood',
    'DoS UDP Flood': 'UDP Flood',
    'DDoS ICMP Flood': 'ICMP Flood',
    'DoS ICMP Flood': 'ICMP Flood',
    'Recon OS Scan': 'Recon Scan',
    'Recon Port Scan': 'Recon Scan',
    'Recon Vulnerability Scan': 'Recon Scan'
}

dataframe_filtered['Attack Name'] = dataframe_filtered['Attack Name'].replace(attack_mapping)

#### 2. Create a small dataframe for building model by grouping by Attack Name

In [3]:
def sample_or_take_all(group):
    sample_size = min(len(group), 2000)
    return group.sample(n=sample_size, random_state=42)

small_df = dataframe_filtered.groupby('Attack Name').apply(sample_or_take_all)
small_df = small_df.reset_index(level=0)

print(small_df.shape)
print(small_df.head())

(16953, 85)
          Attack Name                                   Flow ID  \
6528   Benign Traffic   192.168.137.234-34.173.20.6-55840-443-6   
26781  Benign Traffic          10.0.0.6-10.0.0.254-38889-1883-6   
6594   Benign Traffic         10.0.0.11-10.0.0.254-40489-1883-6   
4932   Benign Traffic  35.83.169.32-192.168.137.210-443-50650-6   
24854  Benign Traffic          10.0.0.3-10.0.0.254-44505-1883-6   

                Src IP  Src Port           Dst IP  Dst Port  Protocol  \
6528   192.168.137.234     55840      34.173.20.6       443         6   
26781         10.0.0.6     38889       10.0.0.254      1883         6   
6594         10.0.0.11     40489       10.0.0.254      1883         6   
4932      35.83.169.32       443  192.168.137.210     50650         6   
24854         10.0.0.3     44505       10.0.0.254      1883         6   

                    Timestamp  Flow Duration  Total Fwd Packet  ...  \
6528   13/09/2023 12:24:22 PM       60363476                17  ...   
2678

In [4]:
small_df['Attack Name'].value_counts()

Attack Name
Benign Traffic             2000
DoS TCP Flood              2000
ICMP Flood                 2000
MQTT DDoS Publish Flood    2000
MQTT DoS Connect Flood     2000
MQTT Malformed             2000
Recon Scan                 2000
UDP Flood                  2000
MQTT DoS Publish Flood      953
Name: count, dtype: int64

In [5]:
small_df.columns

Index(['Attack Name', 'Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Dst Port',
       'Protocol', 'Timestamp', 'Flow Duration', 'Total Fwd Packet',
       'Total Bwd packets', 'Total Length of Fwd Packet',
       'Total Length of Bwd Packet', 'Fwd Packet Length Max',
       'Fwd Packet Length Min', 'Fwd Packet Length Mean',
       'Fwd Packet Length Std', 'Bwd Packet Length Max',
       'Bwd Packet Length Min', 'Bwd Packet Length Mean',
       'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s',
       'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min',
       'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max',
       'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std',
       'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags',
       'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length',
       'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s',
       'Packet Length Min', 'Packet Length Max', 'Packet Length Mean',
       'Packet Le

#### 2a. Different small dataframe

In [6]:
def sample_or_take_all(group):
    sample_size = min(len(group), 1000)
    return group.sample(n=sample_size, random_state=42)

small_df_not_filtered = dataframe_combined.groupby('Attack Name').apply(sample_or_take_all)
small_df_not_filtered = small_df_not_filtered.reset_index(level=0)

#### 3. Prepare Dataframe for SVM training

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
import numpy as numpy

# Separate target from features
targets = small_df_not_filtered['Attack Name']
features = small_df_not_filtered.select_dtypes(include=[numpy.number])

# Divide training and testing
features_train, features_test, target_train, target_test = train_test_split(features, targets, test_size=0.2, random_state=42, stratify=targets)

#### 3a. Drop constant values

In [ ]:
from sklearn.feature_selection import VarianceThreshold

var_selector = VarianceThreshold(threshold=0.0)

features_train_clean = var_selector.fit_transform(features_train)
features_test_clean = var_selector.transform(features_test)

print(f"Features before: {features_train.shape[1]}")
print(f"Features after removing constants: {features_train_clean.shape[1]}")

#### 4. Apply Scaler

In [ ]:
# Scale features
scaler = StandardScaler()
features_train_scaled = scaler.fit_transform(features_train_clean)
features_test_scaled = scaler.transform(features_test_clean)

#### 4a. Apply Selector

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

selector = SelectKBest(score_func=f_classif, k=60)
features_train_kbest = selector.fit_transform(features_train_scaled, target_train)
features_test_kbest = selector.transform(features_test_scaled)

print(f"Reduced feature space from {features_train.shape[1]} down to: {features_train_kbest.shape[1]}")

#### 5. Train Model and perform GridSearch

In [ ]:
# Initialize LinearSVC
# svc = SVC(kernel='poly', degree=3, gamma='scale', C=1.0, class_weight='balanced', random_state=42)

# Train model
# svc.fit(features_train_scaled, target_train)
from sklearn.model_selection import GridSearchCV

param_grid = {
    'kernel': ['rbf'],
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto'],
    'class_weight': [None, 'balanced'],
}
svc = SVC()
grid_search = GridSearchCV(svc, param_grid)
grid_search.fit(features_train_kbest, target_train)

print("best parameters:")
print(grid_search.best_params_)
print("Best Cross-Validation Score:")
print(grid_search.best_score_)

#### 6. Support Vectors

In [ ]:
# Does not work with GridSearchCV
# Total number of support vectors across all classes
print(f"Total Support Vectors: {grid_search.support_vectors_.shape[0]}")

# Number of support vectors per class
for i, class_name in enumerate(svc.classes_):
    print(f"Class '{class_name}': {svc.n_support_[i]} support vectors")

#### 7. Model Evaluation

In [ ]:
# Evaluate the model
target_prediction = grid_search.predict(features_test_kbest)
print(f"Accuracy score: {accuracy_score(target_test, target_prediction):.4f}")
print("Classification Report:")
print(classification_report(target_test, target_prediction, zero_division=0))

#### 8. Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
cm = confusion_matrix(target_test, target_prediction)

fig, ax = plt.subplots(figsize=(10, 10))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=grid_search.classes_)
disp.plot(ax=ax, xticks_rotation='vertical', cmap=plt.cm.Blues)
plt.title("SVM Confusion Matrix")
plt.tight_layout()
plt.show()